# 06. Evaluation and Comparison

This notebook evaluates the six clustering experiments and compares them using internal clustering metrics.

## Purpose
The aim of this notebook is to:
- load the reduced representations used for clustering
- load the predicted labels from all six experiments
- compute internal evaluation metrics
- compare the results in a single summary table

## Metrics used
- **Silhouette Score**: higher values indicate better separation and cohesion
- **Davies-Bouldin Index**: lower values indicate better clustering structure

## Experimental note
The clustering models were developed and tested separately during the dissertation workflow. This notebook provides a clean, consolidated evaluation stage for comparing the final selected experiments.

## Expected inputs
This notebook expects:
- reduced representations:
  - `tfidf_umap_10d.npy`
  - `sbert_umap_10d.npy`
- cluster labels:
  - `hdbscan_tfidf_labels.npy`
  - `hdbscan_sbert_labels.npy`
  - `kmeans_tfidf_labels.npy`
  - `kmeans_sbert_labels.npy`
  - `birch_tfidf_labels.npy`
  - `birch_sbert_labels.npy`




## 1. Import libraries

This section imports the required libraries for loading arrays, computing evaluation metrics, and presenting the final comparison table.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.metrics import silhouette_score, davies_bouldin_score

## 2. Define input files

In [ ]:
tfidf_data_file = Path("tfidf_umap_10d.npy")
sbert_data_file = Path("sbert_umap_10d.npy")

hdbscan_tfidf_file = Path("hdbscan_tfidf_labels.npy")
hdbscan_sbert_file = Path("hdbscan_sbert_labels.npy")

kmeans_tfidf_file = Path("kmeans_tfidf_labels.npy")
kmeans_sbert_file = Path("kmeans_sbert_labels.npy")

birch_tfidf_file = Path("birch_tfidf_labels.npy")
birch_sbert_file = Path("birch_sbert_labels.npy")

## 3. Load reduced representations

In [ ]:
tfidf_umap_10d = np.load(tfidf_data_file)
sbert_umap_10d = np.load(sbert_data_file)

print("TF-IDF reduced shape:", tfidf_umap_10d.shape)
print("S-BERT reduced shape:", sbert_umap_10d.shape)

## 4. Load cluster labels

In [ ]:
hdbscan_tfidf_labels = np.load(hdbscan_tfidf_file)
hdbscan_sbert_labels = np.load(hdbscan_sbert_file)

kmeans_tfidf_labels = np.load(kmeans_tfidf_file)
kmeans_sbert_labels = np.load(kmeans_sbert_file)

birch_tfidf_labels = np.load(birch_tfidf_file)
birch_sbert_labels = np.load(birch_sbert_file)

print("Loaded all cluster label arrays successfully.")

## 5. Define evaluation helper functions

For HDBSCAN, noise points labelled as `-1` are excluded from metric computation because they do not belong to a regular cluster.


In [ ]:
def filter_noise_points(X, labels):
    mask = labels != -1
    return X[mask], labels[mask]

def evaluate_clustering(X, labels, method_name):
    unique_labels = np.unique(labels)

    if len(unique_labels) < 2:
        return {
            "Method": method_name,
            "Clusters": len(unique_labels),
            "Silhouette Score": np.nan,
            "Davies-Bouldin Index": np.nan,
            "Evaluated Samples": len(labels)
        }

    sil = silhouette_score(X, labels)
    dbi = davies_bouldin_score(X, labels)

    return {
        "Method": method_name,
        "Clusters": len(unique_labels),
        "Silhouette Score": round(sil, 4),
        "Davies-Bouldin Index": round(dbi, 4),
        "Evaluated Samples": len(labels)
    }

## 6. Evaluate all six experiments

In [ ]:
results = []

# HDBSCAN + TF-IDF
X_eval, y_eval = filter_noise_points(tfidf_umap_10d, hdbscan_tfidf_labels)
results.append(evaluate_clustering(X_eval, y_eval, "HDBSCAN + TF-IDF"))

# HDBSCAN + S-BERT
X_eval, y_eval = filter_noise_points(sbert_umap_10d, hdbscan_sbert_labels)
results.append(evaluate_clustering(X_eval, y_eval, "HDBSCAN + S-BERT"))

# K-Means + TF-IDF
results.append(evaluate_clustering(tfidf_umap_10d, kmeans_tfidf_labels, "K-Means + TF-IDF"))

# K-Means + S-BERT
results.append(evaluate_clustering(sbert_umap_10d, kmeans_sbert_labels, "K-Means + S-BERT"))

# BIRCH + TF-IDF
results.append(evaluate_clustering(tfidf_umap_10d, birch_tfidf_labels, "BIRCH + TF-IDF"))

# BIRCH + S-BERT
results.append(evaluate_clustering(sbert_umap_10d, birch_sbert_labels, "BIRCH + S-BERT"))

results_df = pd.DataFrame(results)
results_df

## 7. Sort and compare results

The table below provides a clear side-by-side comparison across all six experiments.


In [ ]:
comparison_df = results_df.sort_values(
    by=["Silhouette Score", "Davies-Bouldin Index"],
    ascending=[False, True]
).reset_index(drop=True)

comparison_df

## 8. Save comparison table

This saves the evaluation summary for later reporting or inclusion in the dissertation appendix.


In [ ]:
output_file = "clustering_evaluation_comparison.xlsx"
comparison_df.to_excel(output_file, index=False)

print(f"Saved: {output_file}")

## 9. Notes

This notebook provides a compact internal evaluation summary across all six experiments. In general:
- higher **Silhouette Score** indicates better separation between clusters
- lower **Davies-Bouldin Index** indicates better compactness and separation

The final ranking should be interpreted alongside manual validation and qualitative inspection, since internal metrics alone do not fully capture topical coherence.


## 10. Final summary

At this stage, the six experiments have been evaluated in a single comparison notebook. This creates a clean bridge between:
- clustering
- quantitative comparison
- later interpretation and visualisation
